In [2]:
print("hello")

hello


In [4]:
import pandas as pd

In [5]:
data = pd.read_csv("student_multiclass_dataset.csv")

In [6]:
data.head()

,Student_ID,Age,Gender,Study_Hours_Per_Day,Attendance_Percentage,Assignments_Completed,Previous_Exam_Score,Internet_Usage_Hours,Sleep_Hours,Extra_Curricular,Family_Income,Teacher_Rating,Performance_Level
0,1,22,Male,7.1,56.5,8,34.5,7.9,9.2,No,Low,3.6,Medium
1,2,19,Female,4.9,90.3,12,36.2,7.1,5.0,Yes,High,1.9,Medium
2,3,23,Male,4.6,76.3,14,98.3,6.5,6.2,No,High,3.2,High
3,4,20,Female,2.3,98.9,14,30.4,3.3,8.5,Yes,Medium,1.7,Medium
4,5,22,Male,4.1,82.0,0,78.3,6.4,9.2,Yes,Medium,3.6,Medium


In [7]:
# Encoding Categorical Columns
from sklearn.preprocessing import LabelEncoder

gender_encoder = LabelEncoder()
extra_encoder = LabelEncoder()
income_encoder = LabelEncoder()
target_encoder = LabelEncoder()

data["Gender"] = gender_encoder.fit_transform(data["Gender"])
data["Extra_Curricular"] = extra_encoder.fit_transform(data["Extra_Curricular"])
data["Family_Income"] = income_encoder.fit_transform(data["Family_Income"])
data["Performance_Level"] = target_encoder.fit_transform(data["Performance_Level"])


In [8]:
# Features and Target
X = data.drop("Performance_Level", axis=1)
y = data["Performance_Level"]


In [9]:
print(y.value_counts())

Performance_Level
1    9028
3    6691
0    3451
2     830
Name: count, dtype: int64


In [10]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_smote, y_smote = smote.fit_resample(X, y)

print("After SMOTE")
print(y_smote.value_counts())

After SMOTE
Performance_Level
3    9028
1    9028
0    9028
2    9028
Name: count, dtype: int64


In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_smote,
    y_smote,
    test_size=0.3,
    random_state=42
)

In [12]:
# Feature Scaling
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scale = scaler.fit_transform(X_train)
X_test_scale = scaler.transform(X_test)


In [13]:
from sklearn.tree import DecisionTreeClassifier

clf_entropy = DecisionTreeClassifier(
    criterion="entropy",
    max_depth=10,
    min_samples_leaf=3,
    random_state=100
)
clf_entropy.fit(X_train_scale, y_train)

DecisionTreeClassifier(criterion='entropy', max_depth=10, min_samples_leaf=3,
                       random_state=100)

In [14]:
# Prediction
y_pred = clf_entropy.predict(X_test_scale)

print("Predicted Values:\n", y_pred)


Predicted Values:
 [3 0 0 ... 3 1 0]


In [15]:
# Evaluation
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

print("\nAccuracy:", accuracy_score(y_test, y_pred) * 100)

print("\nClassification Report:\n",
      classification_report(y_test, y_pred))



Confusion Matrix:
 [[2532  165    0    0]
 [ 289 2140    0  286]
 [   0    0 2609   77]
 [   0  291  225 2220]]

Accuracy: 87.69614177589071

Classification Report:
               precision    recall  f1-score   support

           0       0.90      0.94      0.92      2697
           1       0.82      0.79      0.81      2715
           2       0.92      0.97      0.95      2686
           3       0.86      0.81      0.83      2736

    accuracy                           0.88     10834
   macro avg       0.88      0.88      0.88     10834
weighted avg       0.88      0.88      0.88     10834



In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

plt.figure(figsize=(15,10))

plot_tree(
    clf_entropy,
    filled=True,
    rounded=True,
    feature_names=X.columns
)

plt.show()

In [1]:
def manual_predict():
    
    print("\nEnter Student Details:\n")

    student_id = int(input("Student ID: "))
    age = int(input("Age: "))
    gender = input("Gender (Male/Female): ")
    study_hours = float(input("Study Hours: "))
    attendance = float(input("Attendance (%): "))
    assignments = int(input("Assignments Completed: "))
    previous_score = float(input("Previous Score: "))
    internet_usage = float(input("Internet Usage (hrs): "))
    sleep_hours = float(input("Sleep Hours: "))
    extra = input("Extra Curricular (Yes/No): ")
    income = input("Family Income (Low/Medium/High): ")
    rating = float(input("Teacher Rating: "))

    # Encoding (same as training)
    gender = gender_encoder.transform([gender])[0]
    extra = extra_encoder.transform([extra])[0]
    income = income_encoder.transform([income])[0]

    # Create input array
    sample = [[
        student_id,
        age,
        gender,
        study_hours,
        attendance,
        assignments,
        previous_score,
        internet_usage,
        sleep_hours,
        extra,
        income,
        rating
    ]]

    # Scaling
    sample = scaler.transform(sample)

    # Prediction
    pred = clf_entropy.predict(sample)

    # Convert back to original label
    result = target_encoder.inverse_transform(pred)

    print("\n========================")
    print("Predicted Performance:", result[0])
    print("========================\n")

In [2]:
manual_predict()


Enter Student Details:



NameError: name 'gender_encoder' is not defined